# Stage 7: Code Execution and Evaluation

In this stage, we'll learn how to:
1. Execute generated Gurobi code in isolated environments
2. Implement timeout protection (prevents infinite loops)
3. Capture objective values from solver output
4. Compare results with ground truth
5. Calculate accuracy metrics

Key Concept: Safe Code Execution
- Isolated globals: Each execution in separate namespace
- Timeout protection: Signal-based (Linux/Mac) or threading (Windows)
- Error handling: Graceful failures return None
- Output capture: Redirect stdout/stderr to capture solver logs

Output: tutorial_results.csv + accuracy summary

In [1]:
import pickle
import sys
import traceback
import pandas as pd
from io import StringIO
from contextlib import contextmanager
import signal

from config import (
    EXECUTION_TIMEOUT,
    RESULTS_PATH,
    SUMMARY_PATH,
    VERBOSE,
    OPENROUTER_MODEL,
    EMBEDDING_MODEL,
    CROSS_ENCODER_MODEL,
    RETRIEVAL_TOP_K
)

## SECTION 1: TIMEOUT HANDLING

In [2]:
class TimeoutException(Exception):
    """Exception raised when code execution exceeds time limit."""
    pass


@contextmanager
def time_limit(seconds: int):
    """
    Context manager for enforcing execution timeout.

    Uses Unix signals (SIGALRM) to interrupt execution after N seconds.

    Args:
        seconds: Maximum execution time

    Raises:
        TimeoutException: If execution exceeds time limit

    Example:
        >>> with time_limit(10):
        ...     slow_function()  # Will be interrupted after 10 seconds

    Note:
        On Windows, signal.SIGALRM is not available. For Windows compatibility,
        use threading-based timeout (not shown here for simplicity).
    """
    def signal_handler(signum, frame):
        raise TimeoutException(f"Timed out after {seconds} seconds!")

    # Set up signal handler
    signal.signal(signal.SIGALRM, signal_handler)
    signal.alarm(seconds)

    try:
        yield
    finally:
        # Disable alarm
        signal.alarm(0)

## SECTION 2: CODE EXECUTION FUNCTION

In [3]:
def execute_gurobi_code(code_str: str, timeout: int = 120) -> tuple:
    """
    Execute Gurobi code in an isolated environment with timeout protection.

    Security & Isolation:
    - Creates separate namespace (isolated_globals) for each execution
    - Imports are only available within this namespace
    - No access to outer scope variables

    Error Handling:
    - Syntax errors: Caught and returned as error message
    - Runtime errors: Caught and returned with full traceback
    - Timeout: Caught and returned as TimeoutException
    - Infeasibility: Gurobi returns specific status codes

    Args:
        code_str: Python code to execute
        timeout: Maximum execution time in seconds

    Returns:
        Tuple of (result, output):
        - result: Objective value (float) or error message (str) or None
        - output: Captured stdout/stderr (solver logs)

    Example Execution Flow:
        1. Create isolated namespace
        2. Redirect stdout/stderr to capture prints
        3. Execute code string with exec()
        4. Call solve() function from executed code
        5. Return objective value
        6. Restore stdout/stderr

    Common Return Values:
        Success:  (1234.56, "Gurobi Optimizer version...\\nOptimal solution found")
        Failure:  (None, "NameError: variable 'x' not defined")
        Timeout:  (None, "Timeout exceeded")
    """
    # Create clean namespace
    isolated_globals = {}
    result = None
    output = ""

    # Redirect stdout/stderr
    old_stdout = sys.stdout
    old_stderr = sys.stderr
    sys.stdout = sys.stderr = captured_output = StringIO()

    try:
        with time_limit(timeout):
            # Execute code
            exec(code_str, isolated_globals)

            # Check if solve function exists
            if 'solve' not in isolated_globals:
                raise ValueError("No solve() function found in the provided code")

            # Execute the solve function
            result = isolated_globals['solve']()

    except TimeoutException as e:
        result = f"Timeout: {str(e)}"

    except Exception as e:
        error_trace = traceback.format_exc()
        result = f"Execution failed:\n{error_trace}"

    finally:
        # Restore stdout/stderr
        sys.stdout = old_stdout
        sys.stderr = old_stderr
        output = captured_output.getvalue()

    return result, output

## SECTION 3: EVALUATION METRICS

In [4]:
def calculate_accuracy(results: list) -> dict:
    """
    Calculate accuracy metrics from execution results.

    Accuracy Definition:
        A result is correct if:
        int(generated_objective) == int(expected_objective)

    We use int() because:
    - Floating point precision varies slightly
    - Gurobi may return 1234.999999 instead of 1235.0
    - For LP problems, optimal values are often integers

    Args:
        results: List of result dictionaries

    Returns:
        Dictionary with metrics:
        - total: Total problems
        - correct: Correctly solved problems
        - failed: Execution failures (None, errors, timeouts)
        - accuracy: Percentage correct (0-100)

    Example:
        Expected: 1500
        Generated: 1500.0 → Correct (int(1500.0) == 1500)
        Generated: 1499.0 → Incorrect
        Generated: None → Failed (not counted as correct)
    """
    total = len(results)
    correct = 0
    failed = 0

    for result in results:
        expected = result['expected_objective']
        generated = result['execution_result']

        # Check if execution succeeded
        if generated is None or isinstance(generated, str):
            failed += 1
            continue

        # Compare objectives (convert to int for robustness)
        try:
            if int(float(generated)) == int(float(expected)):
                correct += 1
        except (ValueError, TypeError):
            failed += 1

    accuracy = (correct / total * 100) if total > 0 else 0

    return {
        'total': total,
        'correct': correct,
        'failed': failed,
        'accuracy': accuracy
    }

## SECTION 4: EXECUTE PIPELINE

In [5]:
print("=" * 80)
print("STAGE 7: CODE EXECUTION AND EVALUATION")
print("=" * 80)

# Load generated code from Stage 6
print(f"\nLoading generated code from Stage 6...")
with open('tutorial_generated_code.pkl', 'rb') as f:
    generation_results = pickle.load(f)

print(f"   Loaded code for {len(generation_results)} problems\n")

# Store execution results
execution_results = []

# Execute each generated code
print(f"Executing generated code...")
print(f"   Timeout: {EXECUTION_TIMEOUT} seconds per problem\n")

for i, result in enumerate(generation_results, 1):
    print(f"{'─' * 80}")
    print(f"Problem #{result['problem_id']} ({i}/{len(generation_results)})")
    print(f"{'─' * 80}")

    problem_text = result['question']
    if len(problem_text) > 150:
        problem_text = problem_text[:150] + "..."

    print(f"   Problem: {problem_text}")
    print(f"   Expected: {result['expected_objective']}")

    # Execute code
    if result['generated_code']:
        print(f"   Executing code...")

        try:
            exec_result, exec_output = execute_gurobi_code(
                result['generated_code'],
                timeout=EXECUTION_TIMEOUT
            )

            # Check if execution succeeded
            if isinstance(exec_result, (int, float)):
                print(f"   Generated: {exec_result}")

                # Check correctness
                if int(float(exec_result)) == int(float(result['expected_objective'])):
                    print(f"   CORRECT!")
                else:
                    print(f"   INCORRECT (expected {result['expected_objective']})")

            else:
                print(f"   Execution failed: {str(exec_result)[:100]}...")

            execution_results.append({
                **result,
                'execution_result': exec_result,
                'execution_output': exec_output
            })

        except Exception as e:
            print(f"   Unexpected error: {str(e)}")
            execution_results.append({
                **result,
                'execution_result': None,
                'execution_output': str(e)
            })

    else:
        print(f"   No code generated (skipping execution)")
        execution_results.append({
            **result,
            'execution_result': None,
            'execution_output': "No code generated"
        })

    print()

STAGE 7: CODE EXECUTION AND EVALUATION

Loading generated code from Stage 6...
   Loaded code for 2 problems

Executing generated code...
   Timeout: 120 seconds per problem

────────────────────────────────────────────────────────────────────────────────
Problem #0 (1/2)
────────────────────────────────────────────────────────────────────────────────
   Problem: A fishery wants to transport their catch. They can either use local sled dogs or trucks. Local sled dogs can take 100 fish per trip while trucks can t...
   Expected: 3000.0
   Executing code...
   Execution failed: None...

────────────────────────────────────────────────────────────────────────────────
Problem #1 (2/2)
────────────────────────────────────────────────────────────────────────────────
   Problem: An office supply company makes two types of printers: color printers and black and white printers. Different sections of the factory with different te...
   Expected: 5050.0
   Executing code...
   Generated: 5050.0
  

In [6]:
print(f"{'=' * 80}")
print("RESULTS SUMMARY")
print(f"{'=' * 80}\n")

# Calculate metrics
metrics = calculate_accuracy(execution_results)

print("Overall Performance:")
print(f"  Total Problems: {metrics['total']}")
print(f"  Correct: {metrics['correct']}")
print(f"  Incorrect: {metrics['total'] - metrics['correct'] - metrics['failed']}")
print(f"  Failed: {metrics['failed']}")
print(f"  Accuracy: {metrics['accuracy']:.2f}%\n")

# Show individual results
print("Individual Results:")
print(f"{'ID':<5} {'Expected':<12} {'Generated':<12} {'Status':<15}")
print(f"{'-'*5} {'-'*12} {'-'*12} {'-'*15}")

for result in execution_results:
    problem_id = result['problem_id']
    expected = result['expected_objective']
    generated = result['execution_result']

    # Format generated value
    if isinstance(generated, (int, float)):
        gen_str = f"{generated:.1f}"
        # Check if correct
        if int(float(generated)) == int(float(expected)):
            status = "✅ Correct"
        else:
            status = "❌ Incorrect"
    elif generated is None:
        gen_str = "None"
        status = "⚠️  Failed"
    else:
        gen_str = "Error"
        status = "❌ Error"

    print(f"{problem_id:<5} {expected:<12} {gen_str:<12} {status:<15}")

# ========================================================================
# SAVE RESULTS
# ========================================================================

print(f"\nSaving results...")

# Save to CSV
df = pd.DataFrame([{
    'problem_id': r['problem_id'],
    'question': r['question'],
    'expected_objective': r['expected_objective'],
    'generated_code': r['generated_code'],
    'reasoning': r.get('reasoning', None),
    'execution_result': r['execution_result'],
    'execution_output': r['execution_output']
} for r in execution_results])

df.to_csv(RESULTS_PATH, index=False)
print(f"   Results saved to: {RESULTS_PATH}")

# Save summary
with open(SUMMARY_PATH, 'w') as f:
    f.write("CHORUS Tutorial - Execution Summary\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Total Problems: {metrics['total']}\n")
    f.write(f"Correct: {metrics['correct']}\n")
    f.write(f"Failed: {metrics['failed']}\n")
    f.write(f"Accuracy: {metrics['accuracy']:.2f}%\n\n")

    f.write("Model Configuration:\n")
    f.write(f"- LLM: {OPENROUTER_MODEL}\n")
    f.write(f"- Embedding: {EMBEDDING_MODEL}\n")
    f.write(f"- Cross-Encoder: {CROSS_ENCODER_MODEL}\n")
    f.write(f"- Retrieval Top-K: {RETRIEVAL_TOP_K}\n")
    f.write(f"- Timeout: {EXECUTION_TIMEOUT}s\n")

print(f"   Summary saved to: {SUMMARY_PATH}")

RESULTS SUMMARY

Overall Performance:
  Total Problems: 2
  Correct: 1
  Incorrect: 0
  Failed: 1
  Accuracy: 50.00%

Individual Results:
ID    Expected     Generated    Status         
----- ------------ ------------ ---------------
0     3000.0       None         ⚠️  Failed     
1     5050.0       5050.0       ✅ Correct      

Saving results...
   Results saved to: /Users/tasnimahmed/Downloads/tutorial/Standalone/tutorial_results.csv
   Summary saved to: /Users/tasnimahmed/Downloads/tutorial/Standalone/tutorial_summary.txt
